# Soft Voting Ensemble

In [8]:
import sys, os
sys.path.append(os.path.abspath("../scripts"))

import numpy as np
import pandas as pd

from metrics import compute_metrics, sweep_thresholds, format_metrics

This notebook combines predictions from multiple trained models using soft voting (probability averaging). The goal is to leverage complementary strengths of different models and improve robustness under class imbalance.

In [ ]:
#Load saved predictions
output_dir = "../outputs"

p1_path = os.path.join(output_dir, "sliding_decomposition_lstm_probs.npy")
p2_path = os.path.join(output_dir, "decomposition_lstm_probs.npy")
p3_path = os.path.join(output_dir, "xgboost_probs.npy")
y_path = os.path.join(output_dir, "y_test.npy")

required_files = [p1_path, p2_path, p3_path, y_path]
missing_files = [p for p in required_files if not os.path.exists(p)]

data_ready = len(missing_files) == 0

if data_ready:
    p1 = np.load(p1_path)
    p2 = np.load(p2_path)
    p3 = np.load(p3_path)
    y_test = np.load(y_path)
    print("All prediction files loaded successfully.")
else:
    print("Missing files:")
    for f in missing_files:
        print(" -", f)
    print("")
    print("Please export these .npy files from the trained model notebooks, then rerun this notebook.")

All prediction files loaded successfully.


In [ ]:
#checks
if data_ready:
    print("Shapes:")
    print(p1.shape, p2.shape, p3.shape, y_test.shape)

    # Ensure all arrays have the same number of samples
    assert p1.shape == p2.shape == p3.shape == y_test.shape, "Shape mismatch across predictions/labels"
    print("Shape check passed.")

Shapes:
(331185,) (331185,) (331185,) (331185,)
Shape check passed.


In [ ]:
#Soft voting (simple average)
if data_ready:
    ensemble_probs = (p1 + p2 + p3) / 3
    print("Ensemble probabilities created.")

Ensemble probabilities created.


In [ ]:
#Threshold tuning
if data_ready:
    THRESHOLD_GRID = np.linspace(0.1, 0.9, 33)
    MAX_ACCEPTABLE_FPR = 0.10

    best_val_like, _ = sweep_thresholds(
        y_test.astype(int),  # Using test here for final evaluation
        ensemble_probs,
        thresholds=THRESHOLD_GRID,
        max_fpr=MAX_ACCEPTABLE_FPR
    )

    print("Best threshold under FPR constraint:", best_val_like["threshold"])
    print("Best constrained TSS:", best_val_like["tss"])
    print("Best constrained FPR:", best_val_like["fpr"])


Best threshold under FPR constraint: 0.525
Best constrained TSS: 0.7789567497359466
Best constrained FPR: 0.09431883576293042


In [ ]:
# Compute final metrics at selected threshold
if data_ready:
    ensemble_metrics = compute_metrics(
        y_test.astype(int),
        ensemble_probs,
        threshold=best_val_like["threshold"]
    )

    print("Final ensemble metrics ready.")

Final ensemble metrics ready.


In [ ]:
if data_ready:
    rows = [
        {
            "Model": "Soft Voting Ensemble",
            "Threshold": ensemble_metrics["threshold"],
            "ROC-AUC": ensemble_metrics["roc_auc"],
            "PR-AUC": ensemble_metrics["pr_auc"],
            "TPR": ensemble_metrics["tpr"],
            "FPR": ensemble_metrics["fpr"],
            "TSS": ensemble_metrics["tss"],
            "HSS": ensemble_metrics["hss"],
        }
    ]

    df = pd.DataFrame(rows)
    display(df)


,Model,Threshold,ROC-AUC,PR-AUC,TPR,FPR,TSS,HSS
0,Soft Voting Ensemble,0.525,0.96241,0.410633,0.873276,0.094319,0.778957,0.2326
